# Aggregate ERA5 cell-area-weighted pairwise statistics

Pool daily weighted sufficient statistics for one calendar year. The estimand covers HeatStressBench's ERA5 cells selected by `lsm > 0.5`; cell area is represented by exact spherical 0.25° latitude-band weights, and every hourly record is weighted equally within its cell.

In [ ]:
from datetime import date
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import plotly.graph_objects as go

repository = Path("..").resolve()

from liljegren_analysis import pooled_area_weighted_difference_statistics

In [ ]:
database_path = (
    repository / "data" / "daily_pairwise_area_weighted_v1.duckdb"
)
year = 2025
start_date = date(year, 1, 1)
end_date = date(year, 12, 31)

In [ ]:
with duckdb.connect(str(database_path), read_only=True) as connection:
    daily_pairwise = connection.execute(
        """
        SELECT
            result_date,
            implementation_a,
            implementation_b,
            n,
            area_weight_sum,
            area_weighted_mean_difference_c,
            area_weighted_variance_difference,
            area_weighted_mean_absolute_difference_c,
            area_weighted_rmse_c
        FROM daily_pairwise_area_weighted
        WHERE result_date BETWEEN ? AND ?
        ORDER BY result_date, implementation_a, implementation_b
        """,
        [start_date, end_date],
    ).df()

daily_pairwise

## Validate daily coverage

Require complete daily coverage for every persisted implementation pair.

In [ ]:
if daily_pairwise.empty:
    raise ValueError(f"No area-weighted pairwise rows found for {year}")

pair_columns = ["implementation_a", "implementation_b"]
expected_date_count = (end_date - start_date).days + 1
coverage = (
    daily_pairwise.groupby(pair_columns, sort=True, as_index=False)
    .agg(date_count=("result_date", "nunique"))
)
invalid_coverage = coverage.loc[
    coverage["date_count"].ne(expected_date_count)
]
if not invalid_coverage.empty:
    raise ValueError(
        "Incomplete daily coverage:\n"
        + invalid_coverage.to_string(index=False)
    )

coverage

## Pool each implementation pair

Pool by `area_weight_sum`. Difference variance is a descriptive weighted population variance, with within-day and between-day components.

In [ ]:
records = []
for (implementation_a, implementation_b), pair in daily_pairwise.groupby(
    pair_columns,
    sort=True,
):
    statistics = pooled_area_weighted_difference_statistics(
        pair["n"],
        area_weight_sums=pair["area_weight_sum"],
        mean_differences=pair["area_weighted_mean_difference_c"],
        population_variances=(
            pair["area_weighted_variance_difference"]
        ),
        mean_absolute_differences=(
            pair["area_weighted_mean_absolute_difference_c"]
        ),
        root_mean_squared_differences=pair["area_weighted_rmse_c"],
    )
    records.append(
        {
            "implementation_a": implementation_a,
            "implementation_b": implementation_b,
            **statistics,
        }
    )

yearly_pairwise = pd.DataFrame.from_records(records)
yearly_pairwise

## Compare weighted metrics as pairwise matrices

Rows represent `implementation_a`, columns represent `implementation_b`, and signed cells read as row minus column.

In [ ]:
metric_specs = {
    "n": {"label": "Matched observations", "unit": "rows"},
    "area_weight_sum": {
        "label": "Area-weight sum",
        "unit": "relative cell-hours",
    },
    "area_weighted_mean_difference_c": {
        "label": "Area-weighted mean difference",
        "unit": "°C",
        "signed": True,
        "reverse_sign": True,
    },
    "area_weighted_variance_difference": {
        "label": "Area-weighted difference variance",
        "unit": "°C²",
    },
    "area_weighted_stddev_difference_c": {
        "label": "Area-weighted difference standard deviation",
        "unit": "°C",
    },
    "area_weighted_mean_absolute_difference_c": {
        "label": "Area-weighted mean absolute difference",
        "unit": "°C",
    },
    "area_weighted_rmse_c": {
        "label": "Area-weighted RMSE",
        "unit": "°C",
    },
}

implementations = sorted(
    set(yearly_pairwise["implementation_a"])
    | set(yearly_pairwise["implementation_b"])
)
if len(implementations) != 7:
    raise ValueError(
        f"Expected 7 implementations, found {len(implementations)}: "
        f"{implementations}"
    )

metric_matrices = {
    metric: pd.DataFrame(
        np.nan,
        index=implementations,
        columns=implementations,
        dtype=float,
    )
    for metric in metric_specs
}

for row in yearly_pairwise.itertuples(index=False):
    for metric, spec in metric_specs.items():
        value = getattr(row, metric)
        metric_matrices[metric].loc[
            row.implementation_a, row.implementation_b
        ] = value
        metric_matrices[metric].loc[
            row.implementation_b, row.implementation_a
        ] = -value if spec.get("reverse_sign", False) else value

metric_matrices["area_weighted_mean_difference_c"]

In [ ]:
figure = go.Figure()
metrics = list(metric_specs)

for position, metric in enumerate(metrics):
    spec = metric_specs[metric]
    matrix = metric_matrices[metric]
    finite_values = matrix.to_numpy()[np.isfinite(matrix.to_numpy())]
    maximum = max(
        float(np.max(np.abs(finite_values))),
        np.finfo(float).eps,
    )
    signed = spec.get("signed", False)
    display_text = matrix.map(
        lambda value: "" if pd.isna(value) else f"{value:,.3g}"
    )

    figure.add_trace(
        go.Heatmap(
            z=matrix.to_numpy(),
            x=implementations,
            y=implementations,
            text=display_text.to_numpy(),
            texttemplate="%{text}",
            colorscale="RdBu_r" if signed else "YlOrRd",
            zmin=-maximum if signed else 0,
            zmax=maximum,
            zmid=0 if signed else None,
            colorbar={"title": {"text": spec["unit"]}},
            hoverongaps=False,
            hovertemplate=(
                "<b>%{y} − %{x}</b><br>"
                + spec["label"]
                + ": %{z:,.5g} "
                + spec["unit"]
                + "<extra></extra>"
            ),
            visible=position == 0,
        )
    )

figure.update_layout(
    title="ERA5 selected land-cell-area weighted pairwise statistics",
    height=720,
    margin={"l": 120, "r": 80, "t": 140, "b": 100},
    updatemenus=[
        {
            "buttons": [
                {
                    "label": metric_specs[metric]["label"],
                    "method": "update",
                    "args": [
                        {"visible": [candidate == metric for candidate in metrics]},
                    ],
                }
                for metric in metrics
            ],
            "direction": "down",
            "showactive": True,
            "x": 0,
            "xanchor": "left",
            "y": 1.12,
            "yanchor": "top",
        }
    ],
    xaxis={"title": "implementation_b", "side": "bottom"},
    yaxis={
        "title": "implementation_a",
        "autorange": "reversed",
        "scaleanchor": "x",
        "scaleratio": 1,
    },
)

figure.show()